# Simulazione e analisi statistica di una coda

## Effetto della distribuzione del servizio sui tempi di attesa

### Domanda del progetto

A parita' di tempo medio di servizio, una distribuzione esponenziale e una distribuzione uniforme producono gli stessi tempi di attesa e la stessa lunghezza della coda?

Si considera una cassa con arrivi casuali. Il progetto confronta i due tipi di servizio mediante:

- distribuzioni e indicatori statistici;
- probabilita' frequentiste, congiunte e condizionate;
- test di Kolmogorov-Smirnov;
- regressione lineare e correlazione;
- ripetizioni Monte Carlo;
- studio al variare del tasso di arrivo.

Sono utilizzati soltanto modelli e strumenti presenti nelle lezioni. Non vengono introdotte formule specifiche esterne della teoria delle code.


## 1. Librerie e impostazioni grafiche

Il blu identifica sempre il servizio esponenziale; l'arancione identifica sempre il servizio uniforme.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import expon
from scipy.stats import uniform
from scipy import stats

np.random.seed(1234)

colore_exp = '#2878B5'
colore_uni = '#F28E2B'
colore_teoria = '#303030'


## 2. Ipotesi e parametri

Si considera una sola cassa. Arrivano mediamente 10 clienti ogni ora e il servizio dura mediamente 4 minuti.

Gli intervalli tra arrivi consecutivi sono esponenziali: questa ipotesi corrisponde a un processo di arrivi poissoniano, come studiato nelle lezioni.

I due casi confrontati sono:

- servizio esponenziale con media 4 minuti;
- servizio uniforme tra 2 e 6 minuti, ancora con media 4 minuti.

Il numero di clienti, gli arrivi e il tempo medio di servizio restano uguali. Cambia soltanto la variabilita' del servizio.


In [ ]:
tasso_arrivi = 10 / 60
tempo_medio_servizio = 4.0
n_clienti = 15000
n_da_scartare = 1000

print("Arrivi medi ogni ora:", tasso_arrivi * 60)
print("Tempo medio di servizio:", tempo_medio_servizio, "minuti")
print("Clienti simulati:", n_clienti)


## 3. Anteprima delle due distribuzioni del servizio

Prima di costruire la coda si osservano subito le due distribuzioni che saranno confrontate. Gli esempi usano 5000 valori casuali e vengono prodotti rapidamente.


In [ ]:
anteprima_exp = expon.rvs(scale=4, size=5000)
x_exp = np.linspace(0, 24, 500)

plt.figure(figsize=(10, 6))
plt.hist(anteprima_exp, bins=45, density=True,
         color=colore_exp, alpha=0.72,
         edgecolor='white', label='Valori simulati')
plt.plot(x_exp, expon.pdf(x_exp, scale=4),
         color=colore_teoria, linewidth=2.5,
         label='Densita teorica esponenziale')
plt.xlabel('Tempo di servizio [minuti]', fontsize=12)
plt.ylabel('Densita di probabilita', fontsize=12)
plt.title('Distribuzione esponenziale del servizio', fontsize=15)
plt.legend(fontsize=11)
plt.show()


In [ ]:
anteprima_uni = uniform.rvs(loc=2, scale=4, size=5000)
x_uni = np.linspace(0, 8, 500)

plt.figure(figsize=(10, 6))
plt.hist(anteprima_uni, bins=32, density=True,
         color=colore_uni, alpha=0.72,
         edgecolor='white', label='Valori simulati')
plt.plot(x_uni, uniform.pdf(x_uni, loc=2, scale=4),
         color=colore_teoria, linewidth=2.5,
         label='Densita teorica uniforme')
plt.xlabel('Tempo di servizio [minuti]', fontsize=12)
plt.ylabel('Densita di probabilita', fontsize=12)
plt.title('Distribuzione uniforme del servizio', fontsize=15)
plt.xlim(0, 8)
plt.legend(fontsize=11)
plt.show()


### Primo confronto numerico

Le due distribuzioni hanno la stessa media teorica, ma deviazioni standard differenti. Per l'esponenziale la deviazione standard e' uguale alla media; per l'uniforme tra 2 e 6 vale circa 1,155 minuti.


In [ ]:
confronto_servizio_anteprima = pd.DataFrame({
    'modello': ['Esponenziale', 'Uniforme'],
    'media_teorica': [4.0, 4.0],
    'media_simulata': [anteprima_exp.mean(), anteprima_uni.mean()],
    'std_teorica': [4.0, (6-2)/np.sqrt(12)],
    'std_simulata': [anteprima_exp.std(), anteprima_uni.std()],
    'minimo_simulato': [anteprima_exp.min(), anteprima_uni.min()],
    'massimo_simulato': [anteprima_exp.max(), anteprima_uni.max()]
})

confronto_servizio_anteprima


## 4. Funzione che simula la coda

Per ogni cliente il programma registra arrivo, durata del servizio, inizio e fine del servizio, attesa e numero di clienti trovati in coda. I clienti vengono serviti nell'ordine di arrivo.


In [ ]:
def simula_coda(n, tasso_arrivi, tempo_medio_servizio, tipo_servizio):
    intervalli_arrivo = expon.rvs(scale=1/tasso_arrivi, size=n)

    tempi_arrivo = np.zeros(n)
    for i in range(1, n):
        tempi_arrivo[i] = tempi_arrivo[i-1] + intervalli_arrivo[i]

    if tipo_servizio == 'esponenziale':
        tempi_servizio = expon.rvs(scale=tempo_medio_servizio, size=n)
    else:
        tempi_servizio = uniform.rvs(loc=2, scale=4, size=n)

    inizio_servizio = np.zeros(n)
    fine_servizio = np.zeros(n)
    tempi_attesa = np.zeros(n)
    numero_in_coda = np.zeros(n, dtype=int)

    primo_non_uscito = 0

    for i in range(n):
        if i == 0:
            inizio_servizio[i] = tempi_arrivo[i]
        else:
            inizio_servizio[i] = max(tempi_arrivo[i], fine_servizio[i-1])

        tempi_attesa[i] = inizio_servizio[i] - tempi_arrivo[i]
        fine_servizio[i] = inizio_servizio[i] + tempi_servizio[i]

        while (primo_non_uscito < i and
               fine_servizio[primo_non_uscito] <= tempi_arrivo[i]):
            primo_non_uscito = primo_non_uscito + 1

        persone_presenti = i - primo_non_uscito
        numero_in_coda[i] = max(persone_presenti - 1, 0)

    dati = pd.DataFrame({
        'arrivo': tempi_arrivo,
        'servizio': tempi_servizio,
        'inizio_servizio': inizio_servizio,
        'fine_servizio': fine_servizio,
        'attesa': tempi_attesa,
        'numero_in_coda': numero_in_coda
    })

    return dati


## 5. Esecuzione delle simulazioni

Si eliminano i primi 1000 clienti, perché la simulazione parte artificialmente con la cassa vuota.


In [ ]:
dati_exp = simula_coda(n_clienti, tasso_arrivi,
                       tempo_medio_servizio, 'esponenziale')
dati_uni = simula_coda(n_clienti, tasso_arrivi,
                       tempo_medio_servizio, 'uniforme')

dati_exp = dati_exp.iloc[n_da_scartare:]
dati_uni = dati_uni.iloc[n_da_scartare:]

print("Clienti analizzati per ogni modello:", len(dati_exp))
dati_exp.head(10)


## 6. Controllo statistico dei modelli di servizio

Il test di Kolmogorov-Smirnov confronta ogni campione simulato con la distribuzione teorica scelta. L'ipotesi nulla e' che il campione provenga da quella distribuzione.


In [ ]:
D_exp, p_exp = stats.kstest(dati_exp.servizio,
                            expon(scale=4).cdf)
D_uni, p_uni = stats.kstest(dati_uni.servizio,
                            uniform(loc=2, scale=4).cdf)

controllo_modelli = pd.DataFrame({
    'modello': ['Esponenziale', 'Uniforme'],
    'media_teorica': [4.0, 4.0],
    'media_stimata': [dati_exp.servizio.mean(), dati_uni.servizio.mean()],
    'std_teorica': [4.0, (6-2)/np.sqrt(12)],
    'std_stimata': [dati_exp.servizio.std(), dati_uni.servizio.std()],
    'D_KS': [D_exp, D_uni],
    'p_value_KS': [p_exp, p_uni]
})

controllo_modelli


## 7. Analisi descrittiva approfondita delle attese

Oltre a media, mediana e deviazione standard vengono calcolati varianza, quartili, quantili, probabilita' di attesa nulla, probabilita' di superare 5 e 10 minuti, numero medio e massimo di clienti in coda.


In [ ]:
def riassunto(dati, nome):
    return {
        'modello': nome,
        'media': dati.attesa.mean(),
        'mediana': dati.attesa.median(),
        'deviazione_standard': dati.attesa.std(),
        'varianza': dati.attesa.var(),
        'primo_quartile': dati.attesa.quantile(0.25),
        'terzo_quartile': dati.attesa.quantile(0.75),
        'quantile_90': dati.attesa.quantile(0.90),
        'quantile_95': dati.attesa.quantile(0.95),
        'prob_attesa_zero': (dati.attesa == 0).mean(),
        'prob_oltre_5_min': (dati.attesa > 5).mean(),
        'prob_oltre_10_min': (dati.attesa > 10).mean(),
        'numero_medio_in_coda': dati.numero_in_coda.mean(),
        'numero_massimo_in_coda': dati.numero_in_coda.max()
    }

statistiche_attesa = pd.DataFrame([
    riassunto(dati_exp, 'Esponenziale'),
    riassunto(dati_uni, 'Uniforme')
])

statistiche_attesa


### Distribuzioni dei tempi di attesa

I due istogrammi usano gli stessi intervalli e la stessa scala, così il confronto non dipende dalla rappresentazione grafica.


In [ ]:
plt.figure(figsize=(11, 6))
plt.hist(dati_exp.attesa, bins=70, range=(0, 35), density=True,
         color=colore_exp, alpha=0.62, edgecolor='white',
         label='Servizio esponenziale')
plt.hist(dati_uni.attesa, bins=70, range=(0, 35), density=True,
         color=colore_uni, alpha=0.62, edgecolor='white',
         label='Servizio uniforme')
plt.xlabel('Tempo di attesa [minuti]', fontsize=12)
plt.ylabel('Densita di probabilita', fontsize=12)
plt.title('Confronto delle distribuzioni dei tempi di attesa', fontsize=15)
plt.legend(fontsize=11)
plt.show()


In [ ]:
etichette = ['Media', 'Mediana', 'Quantile 90%', 'Quantile 95%']
valori_exp = [dati_exp.attesa.mean(), dati_exp.attesa.median(),
              dati_exp.attesa.quantile(0.90), dati_exp.attesa.quantile(0.95)]
valori_uni = [dati_uni.attesa.mean(), dati_uni.attesa.median(),
              dati_uni.attesa.quantile(0.90), dati_uni.attesa.quantile(0.95)]

x_pos = np.arange(len(etichette))
larghezza = 0.36

plt.figure(figsize=(11, 6))
plt.bar(x_pos-larghezza/2, valori_exp, width=larghezza,
        color=colore_exp, label='Esponenziale')
plt.bar(x_pos+larghezza/2, valori_uni, width=larghezza,
        color=colore_uni, label='Uniforme')
plt.xticks(x_pos, etichette)
plt.ylabel('Tempo [minuti]', fontsize=12)
plt.title('Indicatori dei tempi di attesa', fontsize=15)
plt.legend(fontsize=11)
plt.show()


## 8. Distribuzione del numero di clienti in coda

Le frequenze relative rappresentano la probabilita' empirica di trovare 0, 1, 2, ... clienti in coda.


In [ ]:
valori_coda = np.arange(0, 11)
frequenze_exp = []
frequenze_uni = []

for n in valori_coda:
    frequenze_exp.append((dati_exp.numero_in_coda == n).mean())
    frequenze_uni.append((dati_uni.numero_in_coda == n).mean())

plt.figure(figsize=(11, 6))
plt.bar(valori_coda-0.2, frequenze_exp, width=0.4,
        color=colore_exp, label='Servizio esponenziale')
plt.bar(valori_coda+0.2, frequenze_uni, width=0.4,
        color=colore_uni, label='Servizio uniforme')
plt.xlabel('Numero di clienti in coda', fontsize=12)
plt.ylabel('Probabilita empirica', fontsize=12)
plt.title('Distribuzione del numero di clienti in coda', fontsize=15)
plt.xticks(valori_coda)
plt.legend(fontsize=11)
plt.show()


## 9. Tabella quadrata delle probabilita' congiunte

Si definiscono due eventi:

$$A=\{\text{attesa superiore a 5 minuti}\},$$

$$B=\{\text{almeno 2 clienti trovati in coda}\}.$$

La matrice $2\times2$ contiene quattro eventi incompatibili:

- ne' $A$ ne' $B$;
- $A$ ma non $B$;
- $B$ ma non $A$;
- $A\cap B$.

Le quattro probabilita' sommano a 1. L'intensita' del colore cresce al crescere della probabilita', come nelle tabelle bidimensionali studiate a lezione.


In [ ]:
def matrice_eventi(dati):
    A = dati.attesa > 5
    B = dati.numero_in_coda >= 2

    matrice = np.array([
        [(~A & ~B).mean(), (A & ~B).mean()],
        [(~A & B).mean(),  (A & B).mean()]
    ])

    return matrice


matrice_exp = matrice_eventi(dati_exp)
matrice_uni = matrice_eventi(dati_uni)

print("Somma probabilita esponenziale:", matrice_exp.sum())
print("Somma probabilita uniforme:", matrice_uni.sum())


In [ ]:
def grafico_matrice_probabilita(matrice, titolo):
    fig, ax = plt.subplots(figsize=(7, 7))
    immagine = ax.imshow(matrice, cmap='YlOrRd', vmin=0, vmax=1)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Attesa <= 5 min', 'Attesa > 5 min'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Coda < 2', 'Coda >= 2'])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, '{:.2%}'.format(matrice[i, j]),
                    ha='center', va='center', fontsize=15,
                    color='black')

    ax.set_xlabel('Evento A', fontsize=12)
    ax.set_ylabel('Evento B', fontsize=12)
    ax.set_title(titolo, fontsize=15)
    barra = fig.colorbar(immagine, ax=ax)
    barra.set_label('Probabilita')
    plt.show()


grafico_matrice_probabilita(
    matrice_exp,
    'Probabilita congiunte - servizio esponenziale'
)


In [ ]:
grafico_matrice_probabilita(
    matrice_uni,
    'Probabilita congiunte - servizio uniforme'
)


### Probabilita' marginali, intersezione e probabilita' condizionate

Dalla tabella si ricavano $P(A)$, $P(B)$, $P(A\cap B)$, $P(A|B)$ e $P(B|A)$. Si confronta inoltre $P(A\cap B)$ con $P(A)P(B)$: se fossero uguali, i due eventi sarebbero indipendenti.


In [ ]:
def calcola_probabilita(dati, nome):
    A = dati.attesa > 5
    B = dati.numero_in_coda >= 2

    p_A = A.mean()
    p_B = B.mean()
    p_AB = (A & B).mean()
    p_A_dato_B = p_AB / p_B
    p_B_dato_A = p_AB / p_A

    return {
        'modello': nome,
        'P(A)': p_A,
        'P(B)': p_B,
        'P(A intersezione B)': p_AB,
        'P(A)P(B)': p_A*p_B,
        'P(A dato B)': p_A_dato_B,
        'P(B dato A)': p_B_dato_A
    }


tabella_probabilita = pd.DataFrame([
    calcola_probabilita(dati_exp, 'Esponenziale'),
    calcola_probabilita(dati_uni, 'Uniforme')
])

tabella_probabilita


## 10. Regressione lineare e correlazione

Si propone, per ciascun servizio, il modello

$$\text{attesa}=a+b\,\text{numero in coda}.$$

La pendenza $b$ stima l'aumento medio dell'attesa associato a un cliente aggiuntivo in coda. Il coefficiente $r$ misura la correlazione lineare e $r^2$ la parte di variabilita' descritta dal modello.


In [ ]:
def calcola_regressione(dati, nome):
    risultato = stats.linregress(dati.numero_in_coda, dati.attesa)

    return {
        'modello': nome,
        'intercetta': risultato.intercept,
        'pendenza': risultato.slope,
        'r': risultato.rvalue,
        'r_quadro': risultato.rvalue**2,
        'p_value': risultato.pvalue,
        'errore_pendenza': risultato.stderr
    }


reg_exp = stats.linregress(dati_exp.numero_in_coda, dati_exp.attesa)
reg_uni = stats.linregress(dati_uni.numero_in_coda, dati_uni.attesa)

tabella_regressione = pd.DataFrame([
    calcola_regressione(dati_exp, 'Esponenziale'),
    calcola_regressione(dati_uni, 'Uniforme')
])

tabella_regressione


In [ ]:
valori_regressione = np.arange(0, 11)
medie_coda_exp = []
medie_coda_uni = []

for n in valori_regressione:
    gruppo_exp = dati_exp[dati_exp.numero_in_coda == n]
    gruppo_uni = dati_uni[dati_uni.numero_in_coda == n]
    medie_coda_exp.append(gruppo_exp.attesa.mean())
    medie_coda_uni.append(gruppo_uni.attesa.mean())

retta_exp = reg_exp.intercept + reg_exp.slope * valori_regressione
retta_uni = reg_uni.intercept + reg_uni.slope * valori_regressione

plt.figure(figsize=(11, 6))
plt.plot(valori_regressione, medie_coda_exp, 'o',
         color=colore_exp, markersize=7,
         label='Medie esponenziale')
plt.plot(valori_regressione, retta_exp,
         color=colore_exp, linewidth=2, linestyle='--',
         label='Regressione esponenziale')
plt.plot(valori_regressione, medie_coda_uni, 'o',
         color=colore_uni, markersize=7,
         label='Medie uniforme')
plt.plot(valori_regressione, retta_uni,
         color=colore_uni, linewidth=2, linestyle='--',
         label='Regressione uniforme')
plt.xlabel('Numero di clienti trovati in coda', fontsize=12)
plt.ylabel('Tempo medio di attesa [minuti]', fontsize=12)
plt.title('Attesa e numero di clienti in coda', fontsize=15)
plt.legend(fontsize=10)
plt.show()


## 11. Ripetizioni Monte Carlo e legge dei grandi numeri

La simulazione viene ripetuta molte volte. Per ogni ripetizione si registra l'attesa media. Il grafico mostra la media cumulativa: aumentando il numero di esperimenti, la stima tende a stabilizzarsi.


In [ ]:
def ripeti_simulazioni(n_ripetizioni, n_clienti_mc, tipo_servizio):
    medie = []

    for i in range(n_ripetizioni):
        dati = simula_coda(n_clienti_mc, tasso_arrivi,
                           tempo_medio_servizio, tipo_servizio)
        dati = dati.iloc[300:]
        medie.append(dati.attesa.mean())

    return np.array(medie)


n_ripetizioni = 80
medie_exp = ripeti_simulazioni(n_ripetizioni, 3500, 'esponenziale')
medie_uni = ripeti_simulazioni(n_ripetizioni, 3500, 'uniforme')


In [ ]:
numero_prove = np.arange(1, n_ripetizioni+1)
media_cumulativa_exp = medie_exp.cumsum() / numero_prove
media_cumulativa_uni = medie_uni.cumsum() / numero_prove

plt.figure(figsize=(11, 6))
plt.plot(numero_prove, media_cumulativa_exp,
         color=colore_exp, linewidth=2.2,
         label='Servizio esponenziale')
plt.plot(numero_prove, media_cumulativa_uni,
         color=colore_uni, linewidth=2.2,
         label='Servizio uniforme')
plt.xlabel('Numero di simulazioni considerate', fontsize=12)
plt.ylabel('Media cumulativa dell attesa [minuti]', fontsize=12)
plt.title('Stabilizzazione della stima Monte Carlo', fontsize=15)
plt.legend(fontsize=11)
plt.show()


In [ ]:
media_mc_exp = medie_exp.mean()
media_mc_uni = medie_uni.mean()
std_mc_exp = medie_exp.std()
std_mc_uni = medie_uni.std()

errore_media_exp = std_mc_exp / np.sqrt(n_ripetizioni)
errore_media_uni = std_mc_uni / np.sqrt(n_ripetizioni)

minimo_95_exp = media_mc_exp - 1.96*errore_media_exp
massimo_95_exp = media_mc_exp + 1.96*errore_media_exp
minimo_95_uni = media_mc_uni - 1.96*errore_media_uni
massimo_95_uni = media_mc_uni + 1.96*errore_media_uni

risultati_mc = pd.DataFrame({
    'modello': ['Esponenziale', 'Uniforme'],
    'media_delle_medie': [media_mc_exp, media_mc_uni],
    'std_delle_medie': [std_mc_exp, std_mc_uni],
    'errore_sulla_media': [errore_media_exp, errore_media_uni],
    'limite_95_inferiore': [minimo_95_exp, minimo_95_uni],
    'limite_95_superiore': [massimo_95_exp, massimo_95_uni]
})

risultati_mc


In [ ]:
plt.figure(figsize=(9, 6))
plt.bar([0, 1], [media_mc_exp, media_mc_uni],
        yerr=[1.96*errore_media_exp, 1.96*errore_media_uni],
        capsize=10, color=[colore_exp, colore_uni], width=0.6)
plt.xticks([0, 1], ['Esponenziale', 'Uniforme'])
plt.ylabel('Tempo medio di attesa [minuti]', fontsize=12)
plt.title('Stima Monte Carlo e intervallo al 95%', fontsize=15)
plt.show()


## 12. Studio al variare del tasso di arrivo

Si confrontano 5, 7, 9, 11 e 13 arrivi medi ogni ora, lasciando invariata la durata media del servizio. Per ciascuno scenario si calcolano attesa media, numero medio in coda e probabilita' di superare 5 minuti.


In [ ]:
arrivi_ogni_ora = [5, 7, 9, 11, 13]
studio_arrivi = []

for tipo in ['esponenziale', 'uniforme']:
    for arrivi in arrivi_ogni_ora:
        dati = simula_coda(18000, arrivi/60,
                           tempo_medio_servizio, tipo)
        dati = dati.iloc[1000:]

        studio_arrivi.append({
            'servizio': tipo,
            'arrivi_ora': arrivi,
            'attesa_media': dati.attesa.mean(),
            'numero_medio_in_coda': dati.numero_in_coda.mean(),
            'probabilita_oltre_5_min': (dati.attesa > 5).mean()
        })

studio_arrivi = pd.DataFrame(studio_arrivi)
studio_arrivi


In [ ]:
selezione_exp = studio_arrivi[studio_arrivi.servizio == 'esponenziale']
selezione_uni = studio_arrivi[studio_arrivi.servizio == 'uniforme']

plt.figure(figsize=(11, 6))
plt.plot(selezione_exp.arrivi_ora, selezione_exp.attesa_media,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_exp, label='Esponenziale')
plt.plot(selezione_uni.arrivi_ora, selezione_uni.attesa_media,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_uni, label='Uniforme')
plt.xlabel('Arrivi medi ogni ora', fontsize=12)
plt.ylabel('Tempo medio di attesa [minuti]', fontsize=12)
plt.title('Effetto del tasso di arrivo sull attesa', fontsize=15)
plt.legend(fontsize=11)
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(selezione_exp.arrivi_ora,
         selezione_exp.numero_medio_in_coda,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_exp, label='Esponenziale')
plt.plot(selezione_uni.arrivi_ora,
         selezione_uni.numero_medio_in_coda,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_uni, label='Uniforme')
plt.xlabel('Arrivi medi ogni ora', fontsize=12)
plt.ylabel('Numero medio di clienti in coda', fontsize=12)
plt.title('Effetto del tasso di arrivo sulla coda', fontsize=15)
plt.legend(fontsize=11)
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(selezione_exp.arrivi_ora,
         100*selezione_exp.probabilita_oltre_5_min,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_exp, label='Esponenziale')
plt.plot(selezione_uni.arrivi_ora,
         100*selezione_uni.probabilita_oltre_5_min,
         marker='o', markersize=8, linewidth=2.2,
         color=colore_uni, label='Uniforme')
plt.xlabel('Arrivi medi ogni ora', fontsize=12)
plt.ylabel('Clienti con attesa oltre 5 minuti [%]', fontsize=12)
plt.title('Probabilita di superare 5 minuti', fontsize=15)
plt.legend(fontsize=11)
plt.show()


## 13. Conclusioni da completare dopo l'esecuzione

La conclusione deve riportare i valori numerici principali e discutere:

1. compatibilita' dei tempi di servizio con i modelli mediante test KS;
2. differenze di media, mediana, deviazione standard e quantili delle attese;
3. probabilita' di attesa nulla, oltre 5 minuti e oltre 10 minuti;
4. distribuzione e valore medio del numero di clienti in coda;
5. interpretazione delle quattro caselle delle probabilita' congiunte;
6. probabilita' marginali e condizionate e dipendenza tra gli eventi;
7. pendenza, correlazione, r-quadro e p-value delle regressioni;
8. stabilizzazione delle medie nelle ripetizioni Monte Carlo;
9. intervalli al 95% delle attese medie;
10. variazione di attesa, coda e probabilita' al crescere degli arrivi.

### Interpretazione attesa

I due servizi hanno la stessa durata media, ma l'esponenziale e' piu' variabile e puo' produrre servizi molto lunghi. La simulazione dovrebbe mostrare attese, dispersioni e code maggiori nel caso esponenziale. Aumentando gli arrivi, entrambi i sistemi peggiorano, ma l'effetto dovrebbe essere piu' forte per il servizio esponenziale.

### Limiti

- una sola cassa;
- tasso di arrivo costante;
- clienti serviti nell'ordine di arrivo;
- nessun abbandono della coda;
- arrivi e servizi indipendenti;
- parametri ipotetici, non misurati in una coda reale.


## Corrispondenza con le lezioni

- distribuzioni casuali e Poisson: lezioni 01, 04, 06 e 07;
- Pandas e heatmap: lezioni 03.01 e 03.02;
- indicatori statistici: lezione 04.02;
- probabilita' frequentista: lezione 05.03;
- probabilita' congiunte e condizionate: lezioni 03.03 e 07.02;
- Monte Carlo e legge dei grandi numeri: lezioni 05.03 e 08.01;
- regressione lineare: lezione 15.01;
- intervalli di confidenza: lezione 17.03;
- test di Kolmogorov-Smirnov: lezione 18.03.
